# Real-Time Inference with Unity Catalog Models

This notebook demonstrates the full flow of consuming a shared model via Delta Sharing and deploying it for real-time inference.

Steps covered:

1. **Deploy a Real-Time Model Serving Endpoint**  
   - Programmatically create a model serving endpoint using Databricks REST APIs.
   - Host the shared model as a RESTful API for scalable, production-grade inference.

2. **Perform Inference via REST API**  
   - Send data to the deployed endpoint.
   - Receive predictions programmatically using authentication tokens.

This end-to-end workflow highlights how Delta Sharing and Unity Catalog enable cross-workspace model consumption, deployment, and real-time inferencing — all without relying on manual UI actions.


In [0]:
import requests
import json
import pandas as pd

In [0]:
# Step 1: Define variables
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
WORKSPACE_URL = "https://adb-984752964297111.11.azuredatabricks.net"  # your workspace
headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

# Step 2: Define your serving endpoint config
endpoint_name = "abooth_bike_serving_endpoint"

# Create one serving endpoint with Champion + Challenger
create_payload = {
    "name": endpoint_name,
    "config": {
        "served_models": [
            {
                "name": "champion_model",
                "model_name": "abooth_delta_share.default.bike_sharing_uc_model",
                "model_version": "2", # Champion
                "workload_type": "CPU",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            },
            {
                "name": "challenger_model",
                "model_name": "abooth_delta_share.default.bike_sharing_uc_model",
                "model_version": "3", # Challenger
                "workload_type": "CPU",
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            }
        ],
        "traffic_config": {
            "routes": [
                {
                    "served_model_name": "champion_model",
                    "traffic_percentage": 80
                },
                {
                    "served_model_name": "challenger_model",
                    "traffic_percentage": 20
                }
            ]
        }
    }
}

# Step 3: Make the request to create endpoint
create_url = f"{WORKSPACE_URL}/api/2.0/serving-endpoints"
response = requests.post(create_url, headers=headers, data=json.dumps(create_payload))

# Step 4: Check result
if response.status_code == 200:
    print(f"Successfully created serving endpoint: {endpoint_name}")
else:
    print(f"Failed to create serving endpoint: {response.text}")


Successfully created serving endpoint: abooth_bike_serving_endpoint


## Use Model for Online Inference

In [0]:
# Step 0: Prepare New Data
new_data = pd.DataFrame({
    'season': [1],
    'yr': [0],
    'mnth': [1],
    'hr': [10],
    'holiday': [0],
    'weekday': [3],
    'workingday': [1],
    'weathersit': [1],
    'temp': [0.3],
    'atemp': [0.31],
    'hum': [0.5],
    'windspeed': [0.1]
})

In [0]:
# Step 1: Authenticate to Databricks by retrieving the notebook's API token
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

# Step 2: Define the workspace URL and model serving endpoint URL
WORKSPACE_URL = "https://adb-984752964297111.11.azuredatabricks.net"  # your workspace
ENDPOINT = f"{WORKSPACE_URL}/serving-endpoints/abooth_bike_serving_endpoint/invocations"

# Step 3: Set the HTTP headers, including the authorization token
headers = {
    "Authorization": f"Bearer {DATABRICKS_TOKEN}",
    "Content-Type": "application/json",
}

# Step 4: Build the request payload by converting the input DataFrame to the required format
payload = {"dataframe_split": new_data.to_dict(orient="split")}

# Step 5: Send the POST request to the serving endpoint for inference
response = requests.post(ENDPOINT, headers=headers, data=json.dumps(payload))

# Step 6: Parse and display the prediction results
prediction = response.json()["predictions"][0]

# Look for model info in the response headers
served_model = response.headers.get("served-model-name")

print(f"Prediction: {prediction}")
print(f"Served Model Name: {served_model}")

Prediction: 20.787938243332736
Served Model Name: champion_model


In [0]:
response.headers

{'server': 'databricks', 'content-type': 'application/json', 'served-model-name': 'champion_model', 'endpoint-scale-to-zero': 'True', 'endpoint-small-workload': 'True', 'date': 'Tue, 22 Apr 2025 21:37:23 GMT', 'x-databricks-org-id': '984752964297111', 'x-request-id': '1edc76a0-6cdb-4db9-a545-c2f7d0cc6771', 'content-encoding': 'gzip', 'vary': 'Accept-Encoding', 'strict-transport-security': 'max-age=31536000; includeSubDomains; preload', 'x-content-type-options': 'nosniff', 'alt-svc': 'h3=":5443"; ma=86400, h3-29=":5443"; ma=86400', 'transfer-encoding': 'chunked'}

## ✅ Next Steps

We can extend our workflow to enable **A/B testing**.

In the next section:
- **Send inference requests** and capture which model served each prediction.
- **Log predictions and model metadata** into a centralized Delta table for analysis.
- **Analyze A/B test results** to determine model performance differences.
- **Promote the winning model** by updating the `@prod` alias and adjusting traffic routing.

Let's switch to the *05-ab-testing* notebook.
